# T0.7: GPU gate (L4 x4): seconds per task, vLLM start, quota multiplier

**Before you start:** open your Kaggle profile and write down the GPU quota exactly as shown (for example `0:00 / 30 hrs`), and the time. Best done AFTER the weekly reset so the reading starts near zero.
**Settings:** Add Input = the MAIN competition (this is what makes L4 x4 selectable). Accelerator = GPU L4 x4. Internet = Off.
**Run All** once. Do not restart the server in between (start-up costs about 5 minutes of quota). Stop the session when the last cell finishes.
**After:** wait a few minutes, read the quota again, and send me: before, after, and the `WALL` line printed by the last cell.

In [ ]:
import glob, importlib, json, logging, os, re, shutil, subprocess, sys, time
from pathlib import Path
T_START = time.time()
DATA_DIR = Path("/kaggle/input/competitions/gemma-4-developer-agent")
assert (DATA_DIR / "tasks.jsonl").exists(), "Attach the MAIN competition (Add Input -> Competitions)."
WORKING_DIR = Path("/kaggle/working"); LOG = "/kaggle/working/measurements_gpu.jsonl"
def log(**kw):
    kw["ts_utc"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()); open(LOG, "a").write(json.dumps(kw) + "\n"); print(kw)
def sh(c): return subprocess.run(c, shell=True, capture_output=True, text=True).stdout.strip()
print(sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader"))
log(step="session_start", gpus=sh("nvidia-smi --query-gpu=name --format=csv,noheader").replace("\n", " | "))

for k, v in {"TRANSFORMERS_NO_TF": "1", "TRANSFORMERS_NO_FLAX": "1", "TRANSFORMERS_OFFLINE": "1", "VLLM_NO_USAGE_STATS": "1",
             "VLLM_WORKER_MULTIPROC_METHOD": "spawn", "VLLM_ENGINE_READY_TIMEOUT_S": "1200",
             "VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS": "1", "NCCL_DEBUG": "WARN", "LITELLM_LOCAL_MODEL_COST_MAP": "True",
             "LITELLM_TELEMETRY": "False", "LITELLM_LOG": "WARNING", "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
             "OTEL_SDK_DISABLED": "true", "OTEL_PYTHON_DISABLED": "true"}.items():
    os.environ[k] = v
os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY", "EMPTY"); os.environ["LOCAL_API_KEY"] = os.environ.get("LOCAL_API_KEY", "EMPTY")
for name in ("LiteLLM", "LiteLLM Router", "LiteLLM Proxy", "litellm", "httpx", "httpcore", "openai", "urllib3"):
    logging.getLogger(name).setLevel(logging.WARNING)

# offline wheelhouse (same as the starter notebook)
WHEELHOUSE_DIR = Path("/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse")
assert WHEELHOUSE_DIR.exists(), "Attach the wheelhouse dataset used by the starter notebook: metric/gemma-4-developer-agent-wheelhouse (Add Input -> Datasets)."
import glob as _g
for pat in ("/usr/local/lib/python*/dist-packages/*cutlass*.pth", "/usr/local/lib/python*/site-packages/*cutlass*.pth"):
    for pth in _g.glob(pat):
        try: os.unlink(pth)
        except OSError: pass
tmp_whl = Path("/tmp/wheelhouse"); tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob("*.whl"):
    if "cutlass" in w.name.lower(): continue
    tn = w.name.replace("cu128", "+cu128") if ("cu128" in w.name and "+" not in w.name) else w.name
    tgt = tmp_whl / tn
    if not tgt.exists():
        try: os.symlink(w, tgt)
        except OSError: shutil.copyfile(w, tgt)
wheels = sorted(str(w) for w in tmp_whl.glob("*.whl"))
print(f"Installing {len(wheels)} wheels ...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--force-reinstall", *wheels], check=True)
importlib.invalidate_caches()
print("wheelhouse installed")

In [ ]:
import yaml
from swegemma.models import load_tasks
MODEL_NAME = "gemma-4-e4b-it-qat-w4a16-ct"
AGENT_DIR = WORKING_DIR / "sample_submission"
if AGENT_DIR.exists(): shutil.rmtree(AGENT_DIR)
shutil.copytree(DATA_DIR / "sample_submission", AGENT_DIR)
for yp in [AGENT_DIR / "agent.yaml", *sorted((AGENT_DIR / "sub_agents").glob("*.yaml"))]:
    if yp.is_file():
        t = yp.read_text(encoding="utf-8")
        t = re.sub(r"^(\s*model:\s*).*$", rf"\g<1>{MODEL_NAME}", t, flags=re.M)
        t = re.sub(r"^\s*adapter:\s*.*\n?", "", t, flags=re.M)
        yp.write_text(t, encoding="utf-8")
shutil.rmtree(AGENT_DIR / "adapters", ignore_errors=True)
# budget for the measurement: 10 min per task, 50 tool calls (what you used before)
(AGENT_DIR / "eval_config.yaml").write_text("evaluation:\n  timeout_seconds: 100\n  max_tool_calls: 50\n  max_time_minutes: 10\n  max_turns: 60\n")
TASKS_PATH = DATA_DIR / "tasks.jsonl"; tasks = load_tasks(TASKS_PATH)
IDS = ["requests_6644", "rich_3278", "fastapi_14492"]
SAMPLE_TASKS = [t for t in tasks if t.instance_id in IDS]; print([t.instance_id for t in SAMPLE_TASKS])
GRAPH_DIR = str(DATA_DIR / "graphs"); EMBEDDINGS_DIR = str(DATA_DIR / "embeddings")

In [ ]:
import litellm, torch
from adk_submission import VllmConfig, VllmServer, discover_adapters
from adk_submission.discovery import discover_declared_models
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS
litellm.drop_params = True
declared_model = next(iter(discover_declared_models(AGENT_DIR)))
assert declared_model == MODEL_NAME, declared_model
MODEL_PATH = Path(f"/kaggle/input/models/google/gemma-4/other/{declared_model}/2")
assert MODEL_PATH.exists(), f"Attach the Gemma 4 model input; expected {MODEL_PATH}"
adapters = discover_adapters(str(AGENT_DIR), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
adapter_map = getattr(adapters, "adapters", None) or {}
gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)
cfg = VllmConfig(model=str(MODEL_PATH), port=8000, host="127.0.0.1", tool_call_parser="gemma4", reasoning_parser="gemma4",
                 max_model_len=32768, dtype="bfloat16" if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else "auto",
                 gpu_memory_utilization=0.90, enable_auto_tool_choice=True, enable_lora=bool(adapter_map),
                 max_loras=max(1, len(adapter_map)), max_lora_rank=128, scheduler_reserve_full_isl=False,
                 tensor_parallel_size=tp_size, startup_timeout=60 * 20, extra_args=())
server_instance = VllmServer(cfg, adapter_manifest=adapters)
t0 = time.time(); server_instance.start()
log(step="vllm_start", model=declared_model, tp=tp_size, gpus=gpu_count, seconds=round(time.time() - t0, 1))
log(step="gpu_mem", used=sh("nvidia-smi --query-gpu=memory.used --format=csv,noheader").replace("\n", " | "))
models = server_instance.create_model_registry(aliases=[declared_model], model_prefix="openai/", api_key=os.environ["LOCAL_API_KEY"])

In [ ]:
import asyncio, concurrent.futures
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig, build_submission_limits
from swegemma.evaluate import Evaluator

def run_sync(fn, *a, **kw):
    try: loop = asyncio.get_running_loop()
    except RuntimeError: loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(fn(*a, **kw))).result()
    return asyncio.run(fn(*a, **kw))

ev = yaml.safe_load((AGENT_DIR / "eval_config.yaml").read_text())["evaluation"]
limits, gen_constraints = build_submission_limits()
eval_config = EvalConfig(
    tasks_path=TASKS_PATH, snapshots_dir=DATA_DIR / "snapshots", results_dir=WORKING_DIR / "results", submission_dir=AGENT_DIR,
    models=models, sandbox="subprocess", timeout_seconds=int(ev["timeout_seconds"]), verification_timeout_seconds=300,
    max_time_minutes=float(ev["max_time_minutes"]), max_tool_calls=int(ev["max_tool_calls"]), max_turns=int(ev["max_turns"]),
    limits=limits, generation_constraints=gen_constraints, adapter_manifest=adapters,
    context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
    events_compaction_config=EventsCompactionConfig(compaction_interval=15, overlap_size=2, token_threshold=14336, event_retention_size=5),
    graph_dir=GRAPH_DIR, embeddings_dir=EMBEDDINGS_DIR, wheels_dir=DATA_DIR / "wheels", verbose=False)
evaluator = Evaluator(eval_config)
rows = []
for i, task in enumerate(SAMPLE_TASKS, 1):
    r = run_sync(evaluator.evaluate_task, task=task, task_index=i, total_tasks=len(SAMPLE_TASKS))
    row = dict(step="task", instance=task.instance_id, resolved=bool(r.resolved), exit_code=r.test_exit_code,
               patch_chars=len((r.agent_patch or "").strip()), tool_calls=r.tool_calls, seconds=round(r.duration_seconds, 1))
    rows.append(row); log(**row)
print("tasks:", len(rows), "| resolved:", sum(x["resolved"] for x in rows), "| mean seconds/task:", round(sum(x["seconds"] for x in rows) / max(len(rows), 1), 1))

In [ ]:
log(step="gpu_mem_after", used=sh("nvidia-smi --query-gpu=memory.used --format=csv,noheader").replace("\n", " | "))
try: server_instance.stop()
except Exception as e: print("stop:", e)
wall = (time.time() - T_START) / 3600
log(step="wall", hours=round(wall, 3))
print(f"WALL = {wall:.3f} h ({wall*60:.1f} min). Now STOP the session, wait a few minutes, read the quota again.")
print("Multiplier = (quota_after - quota_before) / WALL.  Download measurements_gpu.jsonl from the Output panel.")